# 01 Field failures: NHTSA complaints and recalls

Pareto by component group and trend by model year for four electric vehicles:
Hyundai Ioniq 5, Ford Mustang Mach-E, Tesla Model 3 and Chevrolet Bolt EV.

Run `python -m src.data.download_nhtsa` first; it writes the inputs to `data/processed/`.

**Counting rule.** A complaint (`ODINO`) can name several components. It counts once in each
component group it names, and Pareto shares are shares of these complaint-group mentions.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # make `src` importable when run from notebooks/

In [ ]:
from src.field import (
    NOT_CODED,
    code_failure_mode,
    complaint_mentions,
    load_failure_mode_rules,
    load_taxonomy,
    map_groups,
    pareto,
    top_n_share,
    trend_by_model_year,
)
from src.plots import pareto_chart, share_trend_chart

PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
UNKNOWN = ["Unknown or other"]

taxonomy = load_taxonomy()
rules = load_failure_mode_rules()
complaints = pd.read_csv(PROCESSED / "nhtsa_complaints.csv", dtype=str, keep_default_na=False)
recalls = pd.read_csv(PROCESSED / "nhtsa_recalls.csv", dtype=str, keep_default_na=False)
print(len(complaints), "complaint-component rows;", complaints["ODINO"].nunique(), "complaints")
print(len(recalls), "recall rows;", recalls["CAMPNO"].nunique(), "campaigns")

## 1. Component taxonomy

NHTSA component labels are long and inconsistent, so the top-level system (text before the
first colon) is mapped to 11 groups plus "Unknown or other". The full mapping is published in
`data/mappings/component_taxonomy.csv`; an unmapped system stops the notebook.

In [ ]:
complaints["group"] = map_groups(complaints["COMPDESC"], taxonomy)
recalls["group"] = map_groups(recalls["COMPNAME"], taxonomy)
mentions = complaint_mentions(complaints, "group")
pd.read_csv(ROOT / "data" / "mappings" / "component_taxonomy.csv")

## 2. Pareto of complaints by component group

In [ ]:
complaint_pareto = pareto(mentions["group"], last=UNKNOWN)
complaint_pareto.to_csv(TABLES / "nhtsa_complaints_pareto.csv", index=False)
top3_complaints = top_n_share(complaint_pareto, 3, last=UNKNOWN)
print(f"{len(mentions):,} complaint-group mentions; top 3 groups: {top3_complaints:.1%}")
pareto_chart(
    complaint_pareto,
    "NHTSA complaints by component group, four EV models",
    FIGURES / "nhtsa_complaints_pareto.png",
    neutral_labels=UNKNOWN,
    unit="complaint-group mentions",
)
complaint_pareto

Unknown or other is drawn last and in gray: it is not a failure mode, but it stays in the
denominator so the shares are not inflated.

In [ ]:
by_vehicle = (
    mentions.pivot_table(index="group", columns="MODELTXT", values="ODINO", aggfunc="nunique")
    .fillna(0)
    .astype(int)
)
by_vehicle.loc[complaint_pareto["label"]]

## 3. Pareto of recall campaigns by component group

A campaign counts once per group it names, across all four vehicles.

In [ ]:
recall_mentions = recalls.drop_duplicates(subset=["CAMPNO", "group"])
recall_pareto = pareto(recall_mentions["group"], last=UNKNOWN)
recall_pareto.to_csv(TABLES / "nhtsa_recalls_pareto.csv", index=False)
top3_recalls = top_n_share(recall_pareto, 3, last=UNKNOWN)
print(f"{recalls['CAMPNO'].nunique()} campaigns; top 3 groups: {top3_recalls:.1%}")
pareto_chart(
    recall_pareto,
    "NHTSA recall campaigns by component group, four EV models",
    FIGURES / "nhtsa_recalls_pareto.png",
    neutral_labels=UNKNOWN,
    unit="campaign-group mentions",
)
recall_pareto

## 4. Electrical failure modes from the complaint narratives

Almost every electrical complaint carries only the label `ELECTRICAL SYSTEM`, so the failure
mode is coded from the narrative (`CDESCR`) with ordered keyword rules published in
`data/mappings/electrical_failure_modes.csv`. The first matching rule wins. Named hardware comes
first (most specific first); symptoms (thermal event, loss of propulsion) come last because they
describe the effect, not the part. Narratives that match no rule stay "Not coded".

In [ ]:
electrical = mentions[mentions["group"] == "Electrical system"].copy()
electrical["failure_mode"] = code_failure_mode(electrical["CDESCR"], rules)
mode_pareto = pareto(electrical["failure_mode"], last=[NOT_CODED])
mode_pareto.to_csv(TABLES / "nhtsa_electrical_failure_modes_pareto.csv", index=False)
top3_modes = top_n_share(mode_pareto, 3, last=[NOT_CODED])
print(f"{len(electrical):,} electrical complaints; top 3 failure modes: {top3_modes:.1%}")
pareto_chart(
    mode_pareto,
    "Electrical complaints by failure mode (coded from narratives)",
    FIGURES / "nhtsa_electrical_failure_modes_pareto.png",
    neutral_labels=[NOT_CODED],
)
mode_pareto

In [ ]:
pd.crosstab(electrical["failure_mode"], electrical["MODELTXT"]).loc[mode_pareto["label"]]

## 5. Trend by model year

Sales volumes are not in the data, so raw complaint counts per model year mostly reflect how
many cars were sold and how long they have been on the road. The chart uses the share of each
model year's complaints that name the electrical system instead. Model years with fewer than
30 complaints are left out of the chart but kept in the table.

In [ ]:
trend = trend_by_model_year(mentions, "group", "Electrical system")
trend.to_csv(TABLES / "nhtsa_electrical_share_by_model_year.csv", index=False)
share_trend_chart(
    trend,
    "Share of complaints naming the electrical system, by model year",
    FIGURES / "nhtsa_electrical_share_by_model_year.png",
)
trend

## 6. Summary for the KPI table

| Source | Top 3 share |
|---|---|
| Complaints by component group | see output below |
| Recall campaigns by component group | see output below |
| Electrical complaints by failure mode | see output below |

In [ ]:
pd.DataFrame(
    {
        "source": ["Complaints by group", "Recall campaigns by group", "Electrical failure modes"],
        "top_3": [
            ", ".join(complaint_pareto.query("label not in @UNKNOWN")["label"].head(3)),
            ", ".join(recall_pareto.query("label not in @UNKNOWN")["label"].head(3)),
            ", ".join(mode_pareto.query("label != @NOT_CODED")["label"].head(3)),
        ],
        "top_3_share": [top3_complaints, top3_recalls, top3_modes],
    }
)